## Neural Network (NN) - `MLPClassifier`

All training code called here.

- Training Time Taken
- Test Accuracy

### Model Parameters

For the Neural Network, the `MLPClassifier` will use one single hidden layer with $20, 100$ nodes, and learning rates of $0.001, 0.01, 0.1$ will be evaluated using cross-validation.

### Load Shared Library

`autoreload` enables it to reload all changes in `shared_library.py`


In [7]:
%load_ext autoreload
%autoreload 2 

# import everything!!
from shared_library import * 

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
RUN_ID = None

### Output Details

Results for neural network are captured in `results/{RUN_ID}/02-mlp/nn_results_adam_relu_{RUN_ID}.csv` and logs for each run are in `results/{RUN_ID}/02-mlp/mlp_experiments_results_{RUN_ID}_{hidden_layer}_{regularization}.log`. The final test fit is written to `results/{RUN_ID}/02-mlp/nn_final_adam_relu_{RUN_ID}.csv`.


### Experiments

Perform a baseline with a smaller set and compare, identify how long it takes to classify a 50-1000 images.

Load a set of 50, 100, 200, 500, 1000 and run a `MLPClassifier` to identify time taken. Do some sample processing to tune `max_iter`, `learning_rate_init` or any other parameters


In [ ]:
RUN_ID = get_run_id(RUN_ID)
RUN_DIR = results_dir(RUN_ID, "02-mlp")
RESULTS_CSV = f"{RUN_DIR}/nn_results_adam_relu_{RUN_ID}.csv"
OUTPUT_LOG = f"{RUN_DIR}/mlp_experiments_results_{RUN_ID}"
N_JOBS = 1

X, y = load_all_images_flat(TRAIN_DATA_PATH, TRAIN_LABEL_PATH)

print(X.shape, X.dtype, X.min(), X.max())
print(y.shape, y.dtype, y.min(), y.max())


def mlp(hidden_layer, regularization):
    print(f"Running for layers:{hidden_layer}, @: {regularization}")

    start_time = perf_counter()

    clf = MLPClassifier(
        solver="adam",
        activation="relu",
        hidden_layer_sizes=hidden_layer,
        max_iter=10000,
        alpha=regularization,
        learning_rate_init=0.0001,
        verbose=True,
        early_stopping=True,
        random_state=SEED,
    )

    # This is only extra line that `cross_val_score` calls, just emulating it. Too lazy to check if skipping `check_scoring` does something extra.
    scoring = check_scoring(estimator=clf, scoring="balanced_accuracy")

    with open(f"{OUTPUT_LOG}_{hidden_layer}_{regularization}.log", "w") as logfile:
        with pipes(stdout=logfile, stderr=logfile):
            print(
                f"====== Running layers: {hidden_layer} @:{regularization} ====== ",
                flush=True,
            )
            scores = cross_validate(
                X=X,
                y=y,
                estimator=clf,
                cv=5,
                scoring=scoring,
                return_estimator=True,
                return_train_score=True,
                n_jobs=N_JOBS,
            )
    end_time = perf_counter()

    test_score = scores["test_score"]
    train_score = scores["train_score"]
    fit_time = scores["fit_time"]
    score_time = scores["score_time"]
    estimator = scores["estimator"]

    return {
        "hidden_layer": hidden_layer,
        "max_iters": 100000,
        "regularization": regularization,
        "test_score": test_score,
        "train_score": train_score,
        "fit_time": fit_time,
        "score_time": score_time,
        # "estimator": estimator,
        "execution_time": end_time - start_time,
    }


hidden_layers = [20, 100]
regularization = [0.0001, 0.001, 0.01, 0.1]

experiments = list(product(hidden_layers, regularization))

# init results
rows = []
df = pd.DataFrame()

print(f"Running {len(experiments)} experiments")

Image Shape:  (5000, 27648) float32 0.0 1.0
(5000, 27648) float32 0.0 1.0
(5000,) uint8 1 10
Running 8 experiments


In [10]:
from IPython.display import clear_output

for la, r in experiments:
    row = mlp(la, r)
    rows.append(row)
    df = pd.DataFrame(rows)
    df.to_csv(RESULTS_CSV, index=False)  # flush to disk immediately
    clear_output(wait=True)
    display(df)

print("This experiment was last executed on: ", datetime.datetime.now())

,hidden_layer,max_iters,regularization,test_score,train_score,fit_time,score_time,execution_time
0,20,100000,0.0001,"[0.194, 0.24499999999999997, 0.255, 0.29500000...","[0.257, 0.29874999999999996, 0.3235, 0.36775, ...","[22.8977530002594, 16.324074029922485, 22.0165...","[0.0724339485168457, 0.09624505043029785, 0.07...",94.555357
1,20,100000,0.0010,"[0.194, 0.24699999999999997, 0.255, 0.29, 0.27...","[0.257, 0.298, 0.32275, 0.34625000000000006, 0...","[12.622570037841797, 18.366766214370728, 25.06...","[0.07048869132995605, 0.06896281242370605, 0.0...",90.963726
2,20,100000,0.0100,"[0.193, 0.248, 0.258, 0.27, 0.24]","[0.25575000000000003, 0.2945, 0.32449999999999...","[18.120239973068237, 18.31508469581604, 22.064...","[0.08868598937988281, 0.08660602569580078, 0.0...",86.568748
3,20,100000,0.1000,"[0.193, 0.24300000000000002, 0.252, 0.28, 0.27...","[0.25549999999999995, 0.295, 0.279999999999999...","[13.314227819442749, 16.633667945861816, 20.20...","[0.08236408233642578, 0.07574892044067383, 0.0...",92.374997
4,100,100000,0.0001,"[0.362, 0.34199999999999997, 0.37, 0.356, 0.366]","[0.50525, 0.5080000000000001, 0.5625, 0.52525,...","[30.086687088012695, 31.135260105133057, 36.36...","[0.10878896713256836, 0.19186902046203613, 0.0...",150.741675
5,100,100000,0.0010,"[0.356, 0.345, 0.332, 0.373, 0.367]","[0.4585, 0.41500000000000004, 0.374, 0.58825, ...","[28.337282180786133, 21.794416189193726, 12.32...","[0.10396218299865723, 0.16770005226135254, 0.0...",126.320240
6,100,100000,0.0100,"[0.366, 0.334, 0.36000000000000004, 0.381, 0.375]","[0.45499999999999996, 0.4205, 0.59024999999999...","[32.116121768951416, 21.883323907852173, 45.80...","[0.08620691299438477, 0.13002419471740723, 0.1...",165.263845
7,100,100000,0.1000,"[0.356, 0.35700000000000004, 0.378, 0.349, 0.359]","[0.47074999999999995, 0.53875, 0.6655, 0.43725...","[27.325373888015747, 35.129658937454224, 58.96...","[0.13286089897155762, 0.09616923332214355, 0.1...",172.443116


This experiment was last executed on:  2026-11-02 09:24:45.045467


#### Training Results with 5-fold cross_val_score

In [13]:
df = pd.read_csv(RESULTS_CSV)
display(df)

for col in ["test_score", "train_score", "fit_time", "score_time"]:
    df[col] = df[col].apply(lambda s: np.array(s.strip(" []").split(), dtype=float))

df["test_score_mean"] = df["test_score"].apply(np.mean)
df["train_score_mean"] = df["train_score"].apply(np.mean)
df["fit_time_mean"] = df["fit_time"].apply(np.mean)
df["score_time_mean"] = df["score_time"].apply(np.mean)
df["execution_time_min"] = df["execution_time"] / 60


display(
    df[
        [
            "hidden_layer",
            "regularization",
            "test_score_mean",
            "train_score_mean",
            "fit_time_mean",
            "score_time_mean",
            "execution_time_min",
        ]
    ]
)

,hidden_layer,max_iters,regularization,test_score,train_score,fit_time,score_time,execution_time
0,20,100000,0.0001,[0.194 0.245 0.255 0.295 0.275],[0.257 0.29875 0.3235 0.36775 0.34075],[22.897753 16.32407403 22.016505 13.553395...,[0.07243395 0.09624505 0.07459807 0.04437399 0...,94.555357
1,20,100000,0.0010,[0.194 0.247 0.255 0.29 0.276],[0.257 0.298 0.32275 0.34625 0.32325],[12.62257004 18.36676621 25.06929779 14.633855...,[0.07048869 0.06896281 0.08500314 0.04866028 0...,90.963726
2,20,100000,0.0100,[0.193 0.248 0.258 0.27 0.24 ],[0.25575 0.2945 0.3245 0.35425 0.2485 ],[18.12023997 18.3150847 22.06471395 18.666369...,[0.08868599 0.08660603 0.03203321 0.03499579 0...,86.568748
3,20,100000,0.1000,[0.193 0.243 0.252 0.28 0.272],[0.2555 0.295 0.28 0.347 0.32275],[13.31422782 16.63366795 20.20245194 20.322371...,[0.08236408 0.07574892 0.08102012 0.04602003 0...,92.374997
4,100,100000,0.0001,[0.362 0.342 0.37 0.356 0.366],[0.50525 0.508 0.5625 0.52525 0.46625],[30.08668709 31.13526011 36.36947894 28.073710...,[0.10878897 0.19186902 0.07928991 0.09078193 0...,150.741675
5,100,100000,0.0010,[0.356 0.345 0.332 0.373 0.367],[0.4585 0.415 0.374 0.58825 0.4625 ],[28.33728218 21.79441619 12.32271123 34.501974...,[0.10396218 0.16770005 0.06403303 0.13735795 0...,126.320240
6,100,100000,0.0100,[0.366 0.334 0.36 0.381 0.375],[0.455 0.4205 0.59025 0.55 0.519 ],[32.11612177 21.88332391 45.80006003 31.005713...,[0.08620691 0.13002419 0.11810088 0.06466603 0...,165.263845
7,100,100000,0.1000,[0.356 0.357 0.378 0.349 0.359],[0.47075 0.53875 0.6655 0.43725 0.51075],[27.32537389 35.12965894 58.96646309 18.836149...,[0.1328609 0.09616923 0.19450688 0.07665491 0...,172.443116


,hidden_layer,regularization,test_score_mean,train_score_mean,fit_time_mean,score_time_mean,execution_time_min
0,20,0.0001,0.2528,0.31755,18.614397,0.062824,1.575923
1,20,0.0010,0.2524,0.30945,17.801562,0.068355,1.516062
2,20,0.0100,0.2418,0.29550,17.023869,0.057033,1.442812
3,20,0.1000,0.2480,0.30005,18.161975,0.064316,1.539583
4,100,0.0001,0.3592,0.51345,29.737699,0.110090,2.512361
5,100,0.0010,0.3546,0.45965,24.625385,0.250029,2.105337
6,100,0.0100,0.3632,0.50695,32.523127,0.121267,2.754397
7,100,0.1000,0.3598,0.52460,34.038820,0.113160,2.874052


### FINAL - Fit Test - running against 8K test data

Model with `hidden_layers` = $100$ and `learning_rate` = $0.01$ has the maximum **Mean Accuracy** of $0.3632$, and we'll use that to do a `fit` test against train data


In [14]:
RESULTS_CSV = f"{RUN_DIR}/nn_final_adam_relu_{RUN_ID}.csv"
N_JOBS = 1

# Load Training Data ~ 5K records
X_train, y_train = load_all_images_flat(TRAIN_DATA_PATH, TRAIN_LABEL_PATH)

print(X_train.shape, X_train.dtype, X_train.min(), X_train.max())
print(y_train.shape, y_train.dtype, y_train.min(), y_train.max())

# Load Test Data ~ 8K records
X_test, y_test = load_all_images_flat(TEST_DATA_PATH, TEST_LABEL_PATH)

print(X_test.shape, X_test.dtype, X_test.min(), X_test.max())
print(y_test.shape, y_test.dtype, y_test.min(), y_test.max())

hidden_layer = 100
regularization = 0.01

start_time = perf_counter()

clf = MLPClassifier(
    solver="adam",
    activation="relu",
    hidden_layer_sizes=hidden_layer,
    max_iter=10000,
    alpha=regularization,
    learning_rate_init=0.0001,
    verbose=True,
    early_stopping=True,
    random_state=SEED,
).fit(X_train, y_train)

# Run an accuracy scoring.
accuracy_score = clf.score(X_test, y_test)

end_time = perf_counter()

output = {
    "hidden_layer": hidden_layer,
    "max_iters": 100000,
    "regularization": regularization,
    "accuracy_score": accuracy_score,
    "execution_time": end_time - start_time,
}

rows = []
df = pd.DataFrame()

rows.append(output)
df = pd.DataFrame(rows)
df.to_csv(RESULTS_CSV, index=False)  # flush to disk immediately
clear_output(wait=True)
display(df)

print("This experiment was last executed on: ", datetime.datetime.now())

,hidden_layer,max_iters,regularization,accuracy_score,execution_time
0,100,100000,0.01,0.371625,40.539824


This experiment was last executed on:  2026-11-02 09:41:10.922484
